### Setting parameters

In [ ]:
# ============================================================
# Neo delta-map runner for Jupyter (no LSF / no file writing)
# - Run top-to-bottom in a notebook cell (or split into cells)
# - iterative_minimization(set_params, fit_params, cholesky_terms) is called per seed
# - Results are stored in `results` (Vector{NamedTuple})
# ============================================================

using PyCall
using NPZ
@pyimport healpy as hp

include("../src/function/neo_delta_map.jl")
include("../src/function/r_estimate_neo.jl")
include("../src/function/set_data_model.jl")

# ----------------------------
# Run settings (edit here)
# ----------------------------
NSIDE = 4
LMIN  = 2
LMAX  = 2 * NSIDE

LMAX_ALM = LMAX
SPIN     = 2

SEEDS_PER_R = 1000
R_VALUES    = [0.01]  # 例: [0.0, 0.001, 0.01]

COV_BASE = "/Users/ikumakiyoshi/Library/Mobile Documents/com~apple~CloudDocs/study_fg_rm/program/julia_Delta_map/Delta_map/make_cmb_covariance_matrix/covariance_matrix"
COV_TAG  = "wo_beam_wo_wl"
FWHM_CON = 0.0

FREQ_BANDS  = [100, 119, 140, 166, 195, 235, 280, 337, 402]  # d1
WHICH_MODEL = "d1"
NUM_I       = 6

MASK_PATH = "../mask/P06_nside_$(NSIDE).fits"

# ----------------------------
# Helpers
# ----------------------------
function load_cov_mats(nside::Int, lmin::Int, lmax::Int; base::String=COV_BASE, tag::String=COV_TAG)
    scal_path = joinpath(base, "paper_smoothing_cov_mat_scal_nside_$(nside)_lmin_$(lmin)_lmax_$(lmax)_$(tag).npy")
    tens_path = joinpath(base, "paper_smoothing_cov_mat_tens_nside_$(nside)_lmin_$(lmin)_lmax_$(lmax)_$(tag).npy")
    return npzread(scal_path), npzread(tens_path)
end

"""
r ごとに「seed に依らない前計算」まで済ませて、
set_params/fit_params のベースと cholesky_terms を返す
"""
function prepare_context_for_r(r_in::Float64;
        nside::Int=NSIDE, lmin::Int=LMIN, lmax::Int=LMAX,
        lmax_alm::Int=LMAX_ALM, spin::Int=SPIN, fwhm_con::Float64=FWHM_CON,
        freq_bands::Vector{Int}=FREQ_BANDS,
        which_model::String=WHICH_MODEL, num_I::Int=NUM_I,
        mask_path::String=MASK_PATH)

    cov_mat_scal, cov_mat_tens = load_cov_mats(nside, lmin, lmax)
    mask = hp.read_map(mask_path)

    # Containers (as in your original code)
    N⁻¹_set       = Matrix{Float64}[]
    TᵀN⁻¹_set     = Matrix{ComplexF64}[]
    TᵀN⁻¹T_set    = Matrix{ComplexF64}[]
    T0_set        = zeros(ComplexF64, 0, 0)
    m_set         = Vector{Float64}[]

    r_est0 = 0.5

    set_params_t = SetParams(freq_bands, which_model, r_in, 2, nside, num_I,
                             cov_mat_scal, cov_mat_tens, mask, m_set,
                             N⁻¹_set, TᵀN⁻¹_set, TᵀN⁻¹T_set, T0_set, spin, lmax_alm)
    fit_params_t = FitParams(-3, 1.5, 20.1, r_est0)

    # seed に依らない前計算
    set_num_I!(set_params_t)
    set_truncate_TᵀN⁻¹_TᵀN⁻¹T!(set_params_t, mask_path; tag=COV_TAG)

    # 元コード互換（これらが global set_params を参照する想定）
    global set_params = set_params_t
    global cholesky_terms = set_cholesky_terms!()
    global matrix_terms   = set_matrix_terms!()

    return set_params_t, fit_params_t, cholesky_terms
end

# ----------------------------
# Main sweep (no file writing)
# ----------------------------
function r_est_for_seed(r_in::Real, seed::Int;
        nside::Int=NSIDE, lmin::Int=LMIN, lmax::Int=LMAX,
        lmax_alm::Int=LMAX_ALM, spin::Int=SPIN, fwhm_con::Float64=FWHM_CON,
        freq_bands::Vector{Int}=FREQ_BANDS,
        which_model::String=WHICH_MODEL, num_I::Int=NUM_I,
        mask_path::String=MASK_PATH)

    # r ごとの前計算（seed非依存） + chol を作る
    set_params, fit_params, chol = prepare_context_for_r(
        Float64(r_in); nside, lmin, lmax, lmax_alm, spin, fwhm_con,
        freq_bands, which_model, num_I, mask_path
    )

    # seed だけ入れて m_vec を作る（seed依存）
    set_params.seed = seed
    set_truncate_m_vec!(set_params, lmin, lmax, fwhm_con)

    # これを回すだけ
    iterative_minimization(set_params, fit_params, chol)

    # 欲しい値だけ返す
    return fit_params.r_est
end

r_est_for_seed (generic function with 1 method)

In [3]:
x = r_est_for_seed(0.01, 7)

Iteration 1: r = 0.013453294960482198, Likelihood = -1.7635617516430428e10
delta_like = 2.7635617516430428e10
delta_r = 0.4865467050395178
Iteration 2: r = 0.011703800421633215, Likelihood = -1.763561753852899e10
delta_like = 22.09856414794922
delta_r = 0.0017494945388489835
Iteration 3: r = 0.011703766268731819, Likelihood = -1.7635617539720467e10
delta_like = 1.1914749145507812
delta_r = 3.415290139585636e-8
Iteration 4: r = 0.011703766268731819, Likelihood = -1.7635617539720467e10
delta_like = 0.0
delta_r = 0.0


0.011703766268731819